# SFJ Media Skills – EVT283: Gesamtdurchgang mit Firecrawl (Einmal-Schlüssel)

**Bedienung:** In Google Colab **Laufzeit → Alle ausführen** wählen. Einmal mit Google anmelden und bei der verdeckten Eingabe den **Firecrawl-API-Schlüssel aus deinem anderen Account** einfügen. **Den Schlüssel nicht direkt in eine Codezelle schreiben.**

Der Lauf arbeitet **ab dem tatsächlichen Google-Drive-Stand EVT282**. Er liest die bereits abgeschlossenen 3’818 Datensätze, 409 Quellenprobleme und 11’841 Feldbelegkandidaten ein. Er wiederholt **nicht** die 2’579 bereits bearbeiteten Quellen. Er prüft bei den verbliebenen zulässigen Fällen 404-Links mit offizieller Anbieter-Domainsuche und andere unvollständige Seiten mit Firecrawl Scrape.

- **Budget:** höchstens **300 reservierte Firecrawl-Credits**, 1 pro einfachem HTML-Scrape und 2 pro Suche mit maximal fünf Ergebnissen. Zusätzliche Kostenmodi wie PDF-Parsing, JSON/LLM-Extraktion, Screenshots, Crawl/Agent und automatische Re-Scrapes sind ausgeschaltet. Die tatsächliche Abrechnung bestimmt Firecrawl; zur Kontrolle das Firecrawl-Dashboard nutzen.
- **Quellenschutz:** ursprüngliche robots-/403-/Sicherheits-/Rate-Limit-Sperren werden nicht umgangen. Neue Quellen müssen aus derselben offiziellen Domain stammen, andernfalls HOLD.
- **Datenqualität:** Einzelkurs ≠ Sammelseite ≠ andere Durchführung ≠ anderes Programm. 307 historische Archiv-/QA09-IDs bleiben geschützt. Keine Feldpromotion ohne individuelle Primärbelege; keine Master-, Website- oder Main-Writes.
- **Wiederaufnahme:** nach je 20 API-Schritten ZIP-Checkpoint mit SHA-256-Rücklesung in Drive. Beim Neustart wird ein vorhandener Prüfschritt nicht erneut berechnet oder verrechnet.

**Wichtig:** Ein einziges Notebook kann die *gesamte Prüfqueue bearbeiten und alle nicht belegbaren Fälle explizit ausweisen*, aber nicht aus fehlenden offiziellen Quellen automatisch belegte Gold-Daten erzeugen. Ohne eine ausdrückliche spätere Release-Freigabe ändert sich nichts an der Website.


In [ ]:
#@title 1. Bibliotheken (keine kostenpflichtigen Firecrawl-SDKs erforderlich)
%pip -q install requests google-api-python-client google-auth-httplib2


In [ ]:
#@title 2. Kontrollfunktionen für Quellen, Kosten, Belege und ZIP-Readback
import csv, io, json, hashlib, re, zipfile, time, socket, ipaddress, getpass
from datetime import datetime, timezone
from collections import Counter, defaultdict
from urllib.parse import urlsplit, urlunsplit
from urllib.robotparser import RobotFileParser
from difflib import SequenceMatcher
import requests

EVT277_FILE='14eVKfokT7AzDGQOQMKRPGA5lK4e_roRz'
EVT277_SHA='d1a4c36ba4a0771f114da18720a5e503702f349b73fb52139e8076a3adefd080'
EVT280_FILE='1Lt8jYi-QQS1KP3E_3ZFZWDOzkm-nGyQV'
EVT280_SHA='f66f81f0ce3ef6b2fbece5d6be187b6a79d8bbc4d70b002385e807d023ac3036'
EVT280_PARENT='1wU2d6AJEpQxFxy22IyJJJ3S7fp9PQgym'
BIBLE='1b1Zx5sjZuqYrEQr6zy9xlxs0eW7R4iDOpEBFl8ZeXsw'
EVT281_DOC='1_EiL8gAVsRycPS93vHQeRkF6-m0vsHg49axfuyGiNe4'
EVT282_DOC='1_mlMzcGi-vCTbLu9J97KPdNtbqiGVsSUObJ2oX1fP2o'
RUN_NAME='SFJ_EVT283_FIRECRAWL_SECRET_PER_RUN_'+EVT277_SHA[:12]
MAX_RESERVED_CREDITS=300
CHECKPOINT_SIZE=20
SOURCE_STATUS_WHITELIST={'HTTP_404','EMPTY_TEXT','CONTENT_INSUFFICIENT','HTTP_202','REDIRECT_INVALID'}
FORBIDDEN_ORIGINAL={'ROBOTS_NOT_ALLOWED_OR_UNKNOWN','REDIRECT_ROBOTS_NOT_ALLOWED',
                    'HTTP_403','DESTINATION_NOT_PUBLIC_OR_UNRESOLVED','HTTP_429','CONTENT_TOO_LARGE'}
USER_AGENT='SFJ-MediaSkills-QualityAudit/1.2 (respectful research, no publication)'

def sha(b):
    return hashlib.sha256(b if isinstance(b,bytes) else b.encode('utf-8')).hexdigest()

def unpack_zipped(data,expected=None):
    if expected and sha(data)!=expected:raise ValueError('SHA-256-Fehler bei Drive-Datei')
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        if z.testzip() is not None:raise ValueError('ZIP-CRC-Fehler')
        return {name:z.read(name) for name in z.namelist()}

def read_tsv(b):
    return list(csv.DictReader(io.StringIO(b.decode('utf-8')),delimiter='\t'))

def safe_cell(v):
    s=json.dumps(v,ensure_ascii=False) if isinstance(v,(list,dict)) else ('' if v is None else str(v))
    return "'"+s if s.lstrip().startswith(('=','+','-','@')) else s

def tsv(rows,fields):
    out=io.StringIO(newline='')
    w=csv.DictWriter(out,fieldnames=fields,delimiter='\t',extrasaction='ignore',lineterminator='\n')
    w.writeheader()
    for r in rows:w.writerow({k:safe_cell(r.get(k,'')) for k in fields})
    return out.getvalue().encode('utf-8')

def pack_zip(files):
    buf=io.BytesIO()
    with zipfile.ZipFile(buf,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=7) as z:
        for k,v in files.items():z.writestr(k,v if isinstance(v,bytes) else v.encode('utf-8'))
    return buf.getvalue()

def get_jsonlines(b):
    return [json.loads(l) for l in b.decode('utf-8').splitlines() if l.strip()]

def strip_www(host):return host.lower().removeprefix('www.')

def same_official_host(left,right):
    try:
        a=strip_www(urlsplit(left).hostname or '')
        b=strip_www(urlsplit(right).hostname or '')
        return bool(a) and a==b
    except (ValueError,AttributeError):return False

def public_https(url):
    try:
        p=urlsplit(url)
        if p.scheme!='https' or not p.hostname or p.username or p.password:return False
        if p.port not in (None,443):return False
        # Reject loopback, private, link-local and unresolvable source endpoints.
        addrs=socket.getaddrinfo(p.hostname,443,type=socket.SOCK_STREAM)
        return bool(addrs) and all(ipaddress.ip_address(a[4][0]).is_global for a in addrs)
    except (OSError,ValueError,TypeError):return False

_robots_cache={}
def robots_publicly_allowed(url):
    p=urlsplit(url)
    if not public_https(url):return False,'DESTINATION_UNSAFE'
    origin=p.scheme+'://'+p.netloc
    if origin in _robots_cache:
        rp=_robots_cache[origin]
        return (bool(rp is True or (hasattr(rp,'can_fetch') and rp.can_fetch('*',url))), 'ROBOT_CHECK_CACHED')
    current=origin+'/robots.txt'
    try:
        for _ in range(3):
            if not public_https(current) or not same_official_host(origin,current):
                _robots_cache[origin]=False;return False,'ROBOTS_REDIRECT_UNSAFE'
            res=requests.get(current,allow_redirects=False,timeout=(6,12),headers={'User-Agent':USER_AGENT},stream=True)
            with res:
                if res.status_code in (301,302,303,307,308):
                    from urllib.parse import urljoin
                    current=urljoin(current,res.headers.get('Location',''))
                    continue
                if res.status_code in (404,410):
                    _robots_cache[origin]=True;return True,'ROBOTS_404'
                if res.status_code!=200:
                    _robots_cache[origin]=False;return False,'ROBOTS_UNAVAILABLE_OR_FORBIDDEN'
                if 'Content-Length' in res.headers and int(res.headers['Content-Length'])>500_000:
                    _robots_cache[origin]=False;return False,'ROBOTS_TOO_LARGE'
                raw=next((chunk for chunk in res.iter_content(chunk_size=500_001) if chunk),b'')
                if len(raw)>500_000:
                    _robots_cache[origin]=False;return False,'ROBOTS_TOO_LARGE'
                rp=RobotFileParser();rp.parse(raw.decode('utf-8','replace').splitlines())
                _robots_cache[origin]=rp
                return rp.can_fetch('*',url),'ROBOTS_RULE'
        _robots_cache[origin]=False;return False,'ROBOTS_REDIRECT_LOOP'
    except Exception:
        _robots_cache[origin]=False;return False,'ROBOTS_FETCH_FAILED'

def prepare_data(original_zip,review_zip):
    a=unpack_zipped(original_zip,EVT277_SHA)
    b=unpack_zipped(review_zip,EVT280_SHA)
    if set(['konsolidierter_3818_grunstock.jsonl'])-set(a):raise ValueError('Original-ZIP unvollständig')
    need={'QUELLEN_409_TRIAGE.tsv','BELEGKANDIDATEN_11841_GATES.tsv','FELDGRUPPEN_REVIEW.tsv','MANIFEST_EVT280.json'}
    if need-set(b):raise ValueError('EVT280-ZIP unvollständig')
    offers=get_jsonlines(a['konsolidierter_3818_grunstock.jsonl'])
    sources=read_tsv(b['QUELLEN_409_TRIAGE.tsv'])
    candidates=read_tsv(b['BELEGKANDIDATEN_11841_GATES.tsv'])
    groups=read_tsv(b['FELDGRUPPEN_REVIEW.tsv'])
    m=json.loads(b['MANIFEST_EVT280.json'])
    if (len(offers),len(sources),len(candidates),len(groups))!=(3818,409,11841,4353):
        raise ValueError('Unzulässige Datenzählerabweichung')
    ids=[x['market']+':'+x['id'] for x in offers]
    if len(set(ids))!=3818:raise ValueError('Doppelte Stable IDs im Quellbestand')
    if sum(x.get('protectedNoReopen') is True for x in offers)<145:
        raise ValueError('Schutzflags unvollständig')
    if len({x['url'] for x in sources})!=409:raise ValueError('Doppelte Quellen-URLs')
    if m['auto_field_writes'] or m['master_or_website_writes'] or m['published']:
        raise ValueError('Unerwartete Schreib-/Release-Daten')
    return offers,sources,candidates,groups

def make_work(sources,offers):
    byid={o['market']+':'+o['id']:o for o in offers}
    work=[]
    for r in sources:
        if r['source_status_after']=='CONTENT_OK':continue
        if r['source_status_before'] in FORBIDDEN_ORIGINAL:continue
        if r['normal_recheck_allowed']!='True':continue
        if r['source_status_after'] not in SOURCE_STATUS_WHITELIST:continue
        ids=json.loads(r['country_ids'])
        if not ids or any(i not in byid or byid[i].get('protectedNoReopen') is True for i in ids):
            raise ValueError('Unbekannte oder geschützte Offer-ID in Recheck')
        offers_here=[byid[i] for i in ids]
        # The original 404 page is unlikely to become a meaningful course page;
        # discover only within its original official host, without crawling a whole site.
        operation='search' if r['source_status_after']=='HTTP_404' else 'scrape'
        work.append({'url':r['url'],'ids':ids,'country':r['country'],
                     'offer_title':offers_here[0].get('title',''),
                     'provider':offers_here[0].get('provider',''),
                     'operation':operation,'reserve_credits':2 if operation=='search' else 1,
                     'priority':r['priority']})
    # Prefer MVP CH/AT/LI and maximise coverage with lower-cost scrapes within each country tier.
    market_rank={'CH':0,'AT':1,'LI':2,'DE':3,'USA':4}
    work.sort(key=lambda x:(market_rank.get(x['country'],9),
           0 if x['operation']=='scrape' else 1,x['url']))
    if len(work)!=232 or Counter(x['operation'] for x in work)!=Counter({'search':135,'scrape':97}):
        raise ValueError('Unbekannte EVT280-Recheck-Abweichung')
    return work

def keyword_norm(s):
    return re.sub(r'\W+',' ',str(s).casefold(),flags=re.UNICODE).strip()

def identity_score(offer_title,page_title):
    a=keyword_norm(offer_title);b=keyword_norm(page_title)
    if not a or not b:return 0.0
    return round(SequenceMatcher(None,a[:180],b[:180]).ratio(),3)

def parse_fc_search_response(payload,original_url,title):
    data=payload.get('data',{}) if isinstance(payload,dict) else {}
    items=(data.get('web',[]) if isinstance(data,dict) else data) or []
    out=[]
    for row in items:
        if not isinstance(row,dict):continue
        target=row.get('url') or (row.get('metadata') or {}).get('url','')
        if not isinstance(target,str) or not target.startswith('https://'):continue
        if not same_official_host(original_url,target):continue
        tit=row.get('title','') or (row.get('metadata') or {}).get('title','')
        out.append({'url':target,'title':tit[:240],
                    'identity_title_similarity':identity_score(title,tit),
                    'snippet':str(row.get('description','') or '')[:1000]})
    return sorted(out,key=lambda x:x['identity_title_similarity'],reverse=True)[:5]

def derive_candidates(candidates,results):
    by_url={x['url']:x for x in results if x.get('url')}
    out=[];counts=Counter()
    for c in candidates:
        rr=by_url.get(c['source_url'],{})
        result_status=rr.get('status','NOT_CALLED_OR_UNAVAILABLE')
        # Duplicate provider pages are never an individual offer-level proof.
        if c['source_offer_count'] and int(c['source_offer_count'])>1:
            route='QUARANTINE_CROSS_OFFER_SOURCE'
        elif result_status=='FC_SCRAPE_OK':
            body=keyword_norm(rr.get('markdown',''))
            needle=keyword_norm(c.get('excerpt',''))
            route='TEXT_REAPPEARS_NOT_FIELD_VERIFIED' if len(needle)>=30 and needle in body else 'EXACT_OFFER_FIELD_REVIEW'
        elif result_status=='FC_SEARCH_RESULTS':
            route='SEARCH_OFFICIAL_CANDIDATE_NOT_FIELD_PROOF'
        else:
            route='SOURCE_PENDING_OR_UNAVAILABLE'
        out.append({**c,'evt283_source_status':result_status,'evt283_review_route':route,
                    'evt283_field_write_allowed':'NO','evt283_website_gold':'HOLD'})
        counts[route]+=1
    return out,dict(counts)


In [ ]:
#@title 3. Google Drive: Startgate, Originaldaten, vorhandene Checkpoints
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload, MediaIoBaseDownload

auth.authenticate_user()
credentials,_=google.auth.default()
drive=build('drive','v3',credentials=credentials,cache_discovery=False)
docs=build('docs','v1',credentials=credentials,cache_discovery=False)

def doc_runs(x):
    if isinstance(x,dict):
        if 'textRun' in x and isinstance(x['textRun'],dict):yield x['textRun'].get('content','')
        for k,v in x.items():
            if k!='textRun':yield from doc_runs(v)
    elif isinstance(x,list):
        for t in x:yield from doc_runs(t)

def docs_text(docid):
    return ''.join(doc_runs(docs.documents().get(documentId=docid,includeTabsContent=True).execute()))

def download_file(fileid):
    b=io.BytesIO();m=MediaIoBaseDownload(b,drive.files().get_media(fileId=fileid),chunksize=1024*1024)
    complete=False
    while not complete:
        _,complete=m.next_chunk(num_retries=2)
    return b.getvalue()

def scan_folder(parent):
    found=[];token=None
    while True:
        kw={'q':"'"+parent+"' in parents and trashed = false",'fields':'nextPageToken,files(id,name,size,mimeType)',
            'pageSize':1000,'orderBy':'name'}
        if token:kw['pageToken']=token
        res=drive.files().list(**kw).execute()
        found.extend(res.get('files',[]));token=res.get('nextPageToken')
        if not token:break
    return found

def drive_folder(parent,name):
    existing=[x for x in scan_folder(parent) if x['name']==name]
    if existing:
        if len(existing)!=1 or existing[0]['mimeType']!='application/vnd.google-apps.folder':
            raise RuntimeError('Mehrdeutiger oder falscher Ausgabeordner')
        return existing[0]['id']
    x=drive.files().create(body={'name':name,'mimeType':'application/vnd.google-apps.folder',
           'parents':[parent]},fields='id,name').execute()
    return x['id']

def immutable_zip_write(parent,name,payload):
    existing=[x for x in scan_folder(parent) if x['name']==name]
    if existing:
        if len(existing)!=1 or sha(download_file(existing[0]['id']))!=sha(payload):
            raise RuntimeError('Checkpoint/ZIP-Namenskonflikt oder abweichender SHA')
        return existing[0]['id']
    request=MediaIoBaseUpload(io.BytesIO(payload),'application/zip',resumable=False)
    f=drive.files().create(body={'name':name,'parents':[parent],
         'mimeType':'application/zip'},media_body=request,fields='id,name,size').execute()
    if sha(download_file(f['id']))!=sha(payload):
        raise RuntimeError('Drive-Checkpoint Readback SHA FEHLER – Lauf stoppen')
    return f['id']

bible=docs_text(BIBLE)
ids=[int(x) for x in re.findall(r'EVT-BIBEL-20261107-(\d+)',bible)]
if not ids or max(ids)<282:raise RuntimeError('Drive-Bibel vor EVT282, STOP')
if '34 FACHLICHEN FELDENTSCHEIDE' not in docs_text(EVT281_DOC):
    raise RuntimeError('EVT281-Nachweise fehlen, STOP')
if 'CH:OFFER-DPSUISSE-002' not in docs_text(EVT282_DOC):
    raise RuntimeError('EVT282-Nachweise fehlen, STOP')
print('Drive Startup-Gate EVT282: PASS, lade ZIPs ...')
offers,sources,candidates,field_groups=prepare_data(download_file(EVT277_FILE),download_file(EVT280_FILE))
work=make_work(sources,offers)
print(f'Basis: {len(offers)} Angebote, {len(sources)} Problem-URLs, {len(candidates)} Belegkandidaten, {len(field_groups)} Feldgruppen')
print(f'Gezielt offen: {len(work)} URLs, darunter {sum(x["operation"]=="search" for x in work)} Anbieter-Suchen und {sum(x["operation"]=="scrape" for x in work)} Scrapes')
out_folder=drive_folder(EVT280_PARENT,RUN_NAME)
print('Ausgabeordner:',f'https://drive.google.com/drive/folders/{out_folder}')

# Never ask the user to paste a key into notebook source. getpass masks it in Colab.
print('\nFirecrawl-API-Schlüssel jetzt im VERDECKTEN Eingabefeld eingeben (von beliebigem Firecrawl-Account).')
FIRECRAWL_API_KEY=getpass.getpass('Firecrawl API-Key (nicht im Notebook speichern): ').strip()
if not FIRECRAWL_API_KEY.startswith('fc-') or len(FIRECRAWL_API_KEY)<12:
    raise RuntimeError('Firecrawl-Key fehlt oder hat unerwartetes Format. Keine API-Aufrufe.')


In [ ]:
#@title 4. Firecrawl: einmalige offizielle Suche/Abrufe und unveränderliche Drive-Checkpoints
# Track planned credits conservatively. No JSON mode, no PDF parser, no enhanced/agent/crawl.
import gc

# Firecrawl Free plans can be limited to five searches/minute. Pace each
# search request instead of getting an immediate 429 and an incomplete run.
_last_search_request_at=0.0
SEARCH_MIN_INTERVAL_SECONDS=13.0

finished={}
files=scan_folder(out_folder)
for f in files:
    if not f['name'].startswith('EVT283_BATCH_') or not f['name'].endswith('.zip'):continue
    z=unpack_zipped(download_file(f['id']))
    if 'records.jsonl' not in z or 'manifest.json' not in z:raise RuntimeError('Fremd-/defekter Checkpoint')
    manifest=json.loads(z['manifest.json'])
    if manifest.get('source_sha256')!=EVT277_SHA or manifest.get('version')!='EVT283_v1':
        raise RuntimeError('Checkpoint mit unzulässigem Parent/Version')
    for rec in get_jsonlines(z['records.jsonl']):
        if rec['url'] in finished:raise RuntimeError('Mehrdeutige Checkpoints zu einer URL')
        finished[rec['url']]=rec
work_by_url={r['url']:r for r in work}
if any(url not in work_by_url for url in finished):raise RuntimeError('Checkpoint-URL ausserhalb Arbeitsliste')
for url,rec in finished.items():
    if rec['ids']!=work_by_url[url]['ids'] or rec['operation']!=work_by_url[url]['operation']:
        raise RuntimeError('Checkpoint-Angebotsidentität verändert')
print(f'Wiederaufnahme: {len(finished)} bereits gespeicherte URLs – keine erneuten API-Aufrufe dafür.')

remaining=[r for r in work if r['url'] not in finished]
used_credits=sum(int(r.get('reserved_credits',0)) for r in finished.values())
if used_credits>MAX_RESERVED_CREDITS:raise RuntimeError('Vorherige Checkpoints überschreiten Laufbudget')
print(f'Noch {len(remaining)} Quellen ohne Checkout, reservierte Credits bisher {used_credits}/{MAX_RESERVED_CREDITS}.')

# Individual HTTP calls make request-count and audit exact; no auto retries to avoid surprise billing.
def firecrawl_call(key,endpoint,payload):
    try:
        resp=requests.post('https://api.firecrawl.dev/v2/'+endpoint,
           headers={'Authorization':'Bearer '+key,'Content-Type':'application/json'},
           json=payload,timeout=(10,75),allow_redirects=False)
        if resp.status_code in (401,403):return 'AUTH_STOP',{}
        if resp.status_code==402:return 'BUDGET_STOP',{}
        if resp.status_code==429:return 'RATE_LIMIT_STOP',{}
        if resp.status_code!=200:return 'FIRECRAWL_HTTP_'+str(resp.status_code),{}
        try:
            data=resp.json()
        except ValueError:
            return 'INVALID_JSON',{}
        return ('OK' if data.get('success') is True else 'FIRECRAWL_RESPONSE_UNSUCCESSFUL'),data
    except requests.RequestException:
        # Deliberately do not show Request/Authorization, key, or source body in logs.
        return 'NETWORK_UNCERTAIN_NO_RETRY',{}

def one_call(item,api_key):
    url=item['url'];op=item['operation'];ids=item['ids']
    base={'url':url,'ids':ids,'country':item['country'],'operation':op,
          'scrape_or_search_completed_at':datetime.now(timezone.utc).isoformat(),
          'reserved_credits':item['reserve_credits'],'status':'',
          'original_url':url,'markdown':'','body_sha256':'','official_candidates':[]}
    # For a Search query, no request to the target site's server is made by us;
    # nevertheless respect the prior 403 / robots guards before provider-specific discovery.
    if op=='scrape':
        allowed,reason=robots_publicly_allowed(url)
        if not allowed:
            base.update(status='POLICY_BLOCKED',reserved_credits=0,note=reason)
            return base
        status,res=firecrawl_call(api_key,'scrape',{'url':url,'formats':['markdown'],
            'onlyMainContent':True,'parsers':[],'maxAge':0,'timeout':30000})
    else:
        if not public_https(url):
            base.update(status='UNSAFE_HOST',reserved_credits=0)
            return base
        host=strip_www(urlsplit(url).hostname)
        query=f'site:{host} {item["offer_title"][:105]}'
        global _last_search_request_at
        wait=max(0,SEARCH_MIN_INTERVAL_SECONDS-(time.monotonic()-_last_search_request_at)) if _last_search_request_at else 0
        if wait:time.sleep(wait)
        _last_search_request_at=time.monotonic()
        status,res=firecrawl_call(api_key,'search',{'query':query,'limit':5,
            'sources':['web'],'includeDomains':[host],'timeout':30000})
    if status!='OK':
        base.update(status=status);return base
    if op=='search':
        hits=parse_fc_search_response(res,url,item['offer_title'])
        base.update(status='FC_SEARCH_RESULTS' if hits else 'NO_MATCHING_OFFICIAL_LINKS',
                    official_candidates=hits)
    else:
        data=res.get('data',{}) if isinstance(res.get('data',{}),dict) else {}
        md=data.get('markdown') or ''
        md=md if isinstance(md,str) else ''
        meta=data.get('metadata') or {}
        source_url=meta.get('sourceURL') or meta.get('url') or url
        if not isinstance(source_url,str) or not same_official_host(url,source_url):
            base.update(status='REDIRECT_ORIGIN_CHANGED_HOLD',redirect_url=str(source_url)[:1000])
        elif len(md.strip())<100:
            base.update(status='SHORT_OR_EMPTY_BODY',markdown=md[:85000])
        else:
            body=md[:85000]
            base.update(status='FC_SCRAPE_OK',markdown=body,body_sha256=sha(body),
                        page_title=str(meta.get('title',''))[:200],final_url=source_url)
    return base

def checkpoint_write(records,first_idx,last_idx):
    if not records:return None
    name=f'EVT283_BATCH_{first_idx:04d}_{last_idx:04d}_'+sha('\n'.join(x['url'] for x in records))[:12]+'.zip'
    data=pack_zip({'records.jsonl':'\n'.join(json.dumps(x,ensure_ascii=False,sort_keys=True) for x in records)+'\n',
         'manifest.json':json.dumps({'source_sha256':EVT277_SHA,'version':'EVT283_v1',
          'urls':len(records),'first_index':first_idx,'last_index':last_idx},ensure_ascii=False,indent=2)})
    return immutable_zip_write(out_folder,name,data)

# Budget is reserved per request BEFORE it is sent; one API call per source.
block=[];start_at=None;stopped_reason='';reserved_at_runstart=used_credits
for idx,item in enumerate(work):
    if item['url'] in finished:continue
    if used_credits+item['reserve_credits']>MAX_RESERVED_CREDITS:
        stopped_reason='CREDIT_CAP_REACHED';break
    if start_at is None:start_at=idx
    r=one_call(item,FIRECRAWL_API_KEY)
    if r['reserved_credits']:
        used_credits+=r['reserved_credits']
    block.append(r);finished[r['url']]=r
    if r['status'] in ('AUTH_STOP','BUDGET_STOP','RATE_LIMIT_STOP'):
        stopped_reason=r['status']
    if len(block)>=CHECKPOINT_SIZE or stopped_reason:
        checkpoint_write(block,start_at,idx)
        print('Readback-Checkpoint',len(finished),'/',len(work),', Credits reserviert',used_credits)
        block=[];start_at=None
    if stopped_reason:break
if block:checkpoint_write(block,start_at,max(i for i,r in enumerate(work) if r['url'] in finished))
del FIRECRAWL_API_KEY
gc.collect()
print('API-Abfragen in diesem Lauf:',len(finished),'- davon aktuell gespeicherte bzw. vorherige Checkpoints.')
print('Budgetverbrauch im Notebook (reservierte Credits, nicht Kontoabrechnung):',used_credits,'/',MAX_RESERVED_CREDITS)
if stopped_reason:print('Stoppgrund:',stopped_reason)


In [ ]:
#@title 5. Alle 409 Quellen und 11’841 Belege zusammenführen, Readback und Abschluss-ZIP
# Every input source row is retained in the final ledger, including protected/no-retry cases.
final_sources=[]
for r in sources:
    done=finished.get(r['url'],{})
    kind=done.get('status','NOT_RECHECKED_OR_POLICY_HOLD')
    final_sources.append({**r,'evt283_status':kind,'evt283_method':done.get('operation',''),
            'evt283_reserved_credits':done.get('reserved_credits',0),
            'evt283_official_candidates':done.get('official_candidates',[]),
            'evt283_new_body_sha256':done.get('body_sha256',''),
            'evt283_field_write_allowed':'NO','evt283_release':'HOLD'})
completed=list(finished.values())
final_candidates,route_stats=derive_candidates(candidates,completed)
work_without_result=[x for x in work if x['url'] not in finished]
report={
 'phase':'EVT283_SAFE_FIRECRAWL_ONE_PASS_READ_ONLY',
 'source_sha256':EVT277_SHA,'evt280_sha256':EVT280_SHA,
 'mvp_countries':['CH','DE','AT','LI','USA'],
 'stable_records':len(offers),'total_source_cases':len(final_sources),
 'total_field_candidates':len(final_candidates),'field_review_groups':len(field_groups),
 'eligible_cases_before_firecrawl':len(work),'attempted_or_resume_covered':len(finished),
 'not_called_due_to_budget_or_stop':len(work_without_result),
 'other_policy_hold_cases':len(final_sources)-len(work),
 'firecrawl_status_counts':dict(Counter(x.get('status','') for x in completed)),
 'field_review_route_counts':route_stats,
 'reserved_credits_total':used_credits,'maximum_reserved_credits':MAX_RESERVED_CREDITS,
 'stopped_reason':stopped_reason,'no_api_key_saved':True,
 'no_master_writes':True,'no_occurrence_writes':True,'no_website_writes':True,
 'no_release':True,'readback_required':True,'as_of':datetime.now(timezone.utc).isoformat()
}
if len(final_sources)!=409 or len(final_candidates)!=11841 or len(field_groups)!=4353:
    raise RuntimeError('Ausgabe hat Zeilen verloren')
if sum(x.get('evt283_field_write_allowed')!='NO' for x in final_sources):
    raise RuntimeError('Feldschreibsperre verletzt')

source_fields=list(final_sources[0].keys())
candidate_fields=list(final_candidates[0].keys())
field_fields=list(field_groups[0].keys())
all_records=''.join(json.dumps(x,ensure_ascii=False,sort_keys=True)+'\n' for x in completed)
readme="""EVT283 – QUALITÄTS-PASS, NICHT RELEASE.\n
Alle 409 EVT280-Quellen und 11841 Kandidaten behalten.\n
Search-Hits sind nur offizielle Nachfolge-URL-KANDIDATEN, kein HTTP-200-/Einzelkursnachweis.\n
Scrapes können als Primärtext dienen, aber Kurs-/Edition-/Varianten-/Tarif-/Ort-/Zugangs-Match bleiben erforderlich.\n
Keine Datenbank-, Master-, GitHub- oder Website-Writes; kein neues Gold-Gate.\n
Der Firecrawl-Key wurde nur zur Laufzeit abgefragt und ist nicht Teil des ZIP.\n"""
bundle=pack_zip({
 'QUELLEN_409_EVT283.tsv':tsv(final_sources,source_fields),
 'BELEGE_11841_EVT283.tsv':tsv(final_candidates,candidate_fields),
 'FELDGRUPPEN_4353_EVT283.tsv':tsv(field_groups,field_fields),
 'FIRECRAWL_ERGEBNISSE.jsonl':all_records,
 'MANIFEST_EVT283.json':json.dumps(report,ensure_ascii=False,sort_keys=True,indent=2),
 'README.txt':readme
})
reopened=unpack_zipped(bundle)
if len(read_tsv(reopened['QUELLEN_409_EVT283.tsv']))!=409 or len(read_tsv(reopened['BELEGE_11841_EVT283.tsv']))!=11841:
    raise RuntimeError('Ausgabe-ZIP nicht konsistent')
finalname='EVT283_ABSCHLUSS_READ_ONLY_'+EVT277_SHA[:12]+'_'+sha(bundle)[:12]+'.zip'
finalfile=immutable_zip_write(out_folder,finalname,bundle)
print('\nEVT283-ERGEBNIS GESPEICHERT UND SHA-READBACK BESTÄTIGT')
print('Drive-Datei:',f'https://drive.google.com/file/d/{finalfile}/view')
print('Drive-Ordner:',f'https://drive.google.com/drive/folders/{out_folder}')
print(json.dumps(report,ensure_ascii=False,indent=2))
print('Wichtig: Alle nicht nachweisbaren Fälle bleiben auf HOLD. Keine Publikation.')


**Nach dem Abschluss:** Gib in ChatGPT nur den Drive-Link zur ausgegebenen `EVT283_ABSCHLUSS_READ_ONLY_...zip` an. Dann können die neu belegbaren Feldentscheidungen kontrolliert gegen den Master geprüft werden. Kein erneuter Voll-Crawl erforderlich. Der API-Key bleibt ausserhalb der Notebook-Datei. 